# DX 704 Week 2 Project

This week's project will analyze fresh strawberry price data for a hypothetical "buy low, freeze, and sell high" business.
Strawberries show strong seasonality in their prices compared to other fruits.

![](https://ers.usda.gov/sites/default/files/_laserfiche/Charts/61401/oct14_finding_plattner_fig01.png)

Image source: https://www.ers.usda.gov/amber-waves/2014/october/seasonal-fresh-fruit-price-patterns-differ-across-commodities-the-case-of-strawberries-and-apples

You are considering a business where you buy strawberries when the prices are very low, carefully freeze them, even more carefully defrost them, and then sell them when the prices are high.
You will forecast strawberry price time series and then use them to tactically pick times to buy, freeze, and sell the strawberries.

The full project description, a template notebook, and raw data are available on GitHub at the following link.

https://github.com/bu-cds-dx704/dx704-project-02


### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Backtest Strawberry Prices

Read the provided "strawberry-prices.tsv" with data from 2020 through 2025.
This data is based on data from the U.S. Bureau of Statistics, but transformed so the ground truth is not online.
https://fred.stlouisfed.org/series/APU0000711415

Use the data for 2020 through 2024 to predict monthly prices in 2025.
Spend some time to make sure you are happy with your methodology and prediction accuracy, since you will reuse the methodology to forecast 2026 next.
Save the 2025 backtest predictions as "strawberry-backtest.tsv" with columns month and price.

In [2]:
#Hint: beware of missing rows of data.
#The source is missing a few months!

In [10]:
# YOUR CHANGES HERE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load data
df = pd.read_csv("strawberry-prices.tsv", sep="\t")

print(df.head())
print(df.columns)
print(df.shape)


        month  price
0  2020-01-01  4.049
1  2020-02-01  3.625
2  2020-03-01  3.377
3  2020-05-01  3.126
4  2020-06-01  2.926
Index(['month', 'price'], dtype='str')
(68, 2)


In [11]:
df["month"] = pd.to_datetime(df["month"])
df = df.sort_values("month").reset_index(drop=True)

print(df.head())
print(df.tail())

       month  price
0 2020-01-01  4.049
1 2020-02-01  3.625
2 2020-03-01  3.377
3 2020-05-01  3.126
4 2020-06-01  2.926
        month  price
63 2025-06-01  3.190
64 2025-07-01  3.187
65 2025-08-01  3.430
66 2025-09-01  3.645
67 2025-12-01  4.988


In [12]:
full_months = pd.date_range(
    start=df["month"].min(),
    end=df["month"].max(),
    freq="MS"
)

missing_months = full_months.difference(df["month"])

print("Missing months:")
print(missing_months)
print("\nNumber missing:", len(missing_months))

Missing months:
DatetimeIndex(['2020-04-01', '2021-12-01', '2025-10-01', '2025-11-01'], dtype='datetime64[us]', freq=None)

Number missing: 4


In [13]:
df = (
    df.set_index("month")
      .reindex(full_months)
      .rename_axis("month")
      .reset_index()
)

print(df[df["price"].isna()])

        month  price
3  2020-04-01    NaN
23 2021-12-01    NaN
69 2025-10-01    NaN
70 2025-11-01    NaN


In [14]:
df = df.set_index("month")

df["price"] = df["price"].interpolate(method="time")

df = df.reset_index()

print("Missing values remaining:")
print(df.isna().sum())

Missing values remaining:
month    0
price    0
dtype: int64


In [15]:
print(df[df["month"].isin(missing_months)])

        month     price
3  2020-04-01  3.249443
23 2021-12-01  4.320213
69 2025-10-01  4.087747
70 2025-11-01  4.545253


In [16]:
print("Total rows after filling missing months:", len(df))
print("Duplicate months:", df["month"].duplicated().sum())
print("Missing prices:", df["price"].isna().sum())

print(df.head())
print(df.tail())

Total rows after filling missing months: 72
Duplicate months: 0
Missing prices: 0
       month     price
0 2020-01-01  4.049000
1 2020-02-01  3.625000
2 2020-03-01  3.377000
3 2020-04-01  3.249443
4 2020-05-01  3.126000
        month     price
67 2025-08-01  3.430000
68 2025-09-01  3.645000
69 2025-10-01  4.087747
70 2025-11-01  4.545253
71 2025-12-01  4.988000


In [17]:
full_months = pd.date_range(
    start=df["month"].min(),
    end=df["month"].max(),
    freq="MS"
)

missing_months = full_months.difference(df["month"])

print("Missing months:")
for month in missing_months:
    print(month.strftime("%Y-%m"))

print("\nNumber missing:", len(missing_months))

Missing months:

Number missing: 0


Please use the same format for the month column as in the training data, i.e. YYYY-MM-01.
The autograder may not be able to parse other formats.

Submit "strawberry-backtest.tsv" in Gradescope.

## Part 2: Backtest Errors

What are the mean and standard deviation of the residuals between your backtest predictions and the ground truth?

Write the mean and standard deviation to a file "backtest-accuracy.tsv" with two columns, mean and std.

In [20]:
train = df[df["month"].dt.year <= 2024].copy()

print(train.shape)
print(train.head())
print(train.tail())

(60, 2)
       month     price
0 2020-01-01  4.049000
1 2020-02-01  3.625000
2 2020-03-01  3.377000
3 2020-04-01  3.249443
4 2020-05-01  3.126000
        month  price
55 2024-08-01  3.347
56 2024-09-01  3.742
57 2024-10-01  3.718
58 2024-11-01  4.420
59 2024-12-01  4.856


In [21]:
model = ExponentialSmoothing(
    train["price"],
    trend="add",
    seasonal="add",
    seasonal_periods=12
)

fit = model.fit()

forecast_2025 = fit.forecast(12)

print(forecast_2025)

60    4.667437
61    4.291835
62    3.865636
63    3.914524
64    3.634634
65    3.381634
66    3.344632
67    3.627428
68    3.780026
69    4.050425
70    4.568427
71    4.968665
dtype: float64


In [22]:
forecast_months = pd.date_range(
    start="2025-01-01",
    periods=12,
    freq="MS"
)

backtest = pd.DataFrame({
    "month": forecast_months,
    "price": forecast_2025.values
})

backtest

,month,price
0,2025-01-01,4.667437
1,2025-02-01,4.291835
2,2025-03-01,3.865636
3,2025-04-01,3.914524
4,2025-05-01,3.634634
5,2025-06-01,3.381634
6,2025-07-01,3.344632
7,2025-08-01,3.627428
8,2025-09-01,3.780026
9,2025-10-01,4.050425


In [23]:
backtest.to_csv(
    "strawberry-backtest.tsv",
    sep="\t",
    index=False
)

print("Saved strawberry-backtest.tsv")

Saved strawberry-backtest.tsv


In [24]:
# YOUR CHANGES HERE
actual = pd.read_csv("strawberry-prices.tsv", sep="\t")

predicted = pd.read_csv("strawberry-backtest.tsv", sep="\t")

actual["month"] = pd.to_datetime(actual["month"])
predicted["month"] = pd.to_datetime(predicted["month"])

actual_2025 = actual[actual["month"].dt.year == 2025].copy()

comparison = predicted.merge(
    actual_2025,
    on="month",
    how="inner",
    suffixes=("_predicted", "_actual")
)

comparison

,month,price_predicted,price_actual
0,2025-01-01,4.667437,4.584
1,2025-02-01,4.291835,4.077
2,2025-03-01,3.865636,3.369
3,2025-04-01,3.914524,3.518
4,2025-05-01,3.634634,3.416
5,2025-06-01,3.381634,3.190
6,2025-07-01,3.344632,3.187
7,2025-08-01,3.627428,3.430
8,2025-09-01,3.780026,3.645
9,2025-12-01,4.968665,4.988


In [25]:
actual["month"] = pd.to_datetime(actual["month"])

actual_2025 = actual[
    actual["month"].dt.year == 2025
].copy()

print(actual_2025)
print("Actual 2025 observations:", len(actual_2025))

        month  price
58 2025-01-01  4.584
59 2025-02-01  4.077
60 2025-03-01  3.369
61 2025-04-01  3.518
62 2025-05-01  3.416
63 2025-06-01  3.190
64 2025-07-01  3.187
65 2025-08-01  3.430
66 2025-09-01  3.645
67 2025-12-01  4.988
Actual 2025 observations: 10


In [26]:
comparison = backtest.merge(
    actual_2025,
    on="month",
    how="inner",
    suffixes=("_predicted", "_actual")
)

comparison

,month,price_predicted,price_actual
0,2025-01-01,4.667437,4.584
1,2025-02-01,4.291835,4.077
2,2025-03-01,3.865636,3.369
3,2025-04-01,3.914524,3.518
4,2025-05-01,3.634634,3.416
5,2025-06-01,3.381634,3.190
6,2025-07-01,3.344632,3.187
7,2025-08-01,3.627428,3.430
8,2025-09-01,3.780026,3.645
9,2025-12-01,4.968665,4.988


In [27]:
comparison["residual"] = (
    comparison["price_actual"]
    - comparison["price_predicted"]
)

comparison

,month,price_predicted,price_actual,residual
0,2025-01-01,4.667437,4.584,-0.083437
1,2025-02-01,4.291835,4.077,-0.214835
2,2025-03-01,3.865636,3.369,-0.496636
3,2025-04-01,3.914524,3.518,-0.396524
4,2025-05-01,3.634634,3.416,-0.218634
5,2025-06-01,3.381634,3.190,-0.191634
6,2025-07-01,3.344632,3.187,-0.157632
7,2025-08-01,3.627428,3.430,-0.197428
8,2025-09-01,3.780026,3.645,-0.135026
9,2025-12-01,4.968665,4.988,0.019335


In [28]:
residual_mean = comparison["residual"].mean()
residual_std = comparison["residual"].std()

print("Number of residuals:", len(comparison))
print("Mean residual:", residual_mean)
print("Standard deviation:", residual_std)

Number of residuals: 10
Mean residual: -0.2072451722685833
Standard deviation: 0.14698638142689274


In [29]:
accuracy = pd.DataFrame({
    "mean": [residual_mean],
    "std": [residual_std]
})

accuracy.to_csv(
    "backtest-accuracy.tsv",
    sep="\t",
    index=False
)

accuracy

,mean,std
0,-0.207245,0.146986


Hint: If the mean residual in your backtest is not close to zero, then your model is likely missing a systematic change and you should go back to improve it.

Submit "backtest-accuracy.tsv" in Gradescope.

## Part 3: Forecast Strawberry Prices

Use all the data from 2020 through 2025 to predict monthly prices in 2026 using the same methodology from part 1.
Make a monthly forecast for each month of 2026 and save it as "strawberry-forecast.tsv" with columns for month and price.


In [30]:
# YOUR CHANGES HERE
train_2026 = df[
    (df["month"].dt.year >= 2020) &
    (df["month"].dt.year <= 2025)
].copy()

print(train_2026.shape)
print(train_2026.head())
print(train_2026.tail())


(72, 2)
       month     price
0 2020-01-01  4.049000
1 2020-02-01  3.625000
2 2020-03-01  3.377000
3 2020-04-01  3.249443
4 2020-05-01  3.126000
        month     price
67 2025-08-01  3.430000
68 2025-09-01  3.645000
69 2025-10-01  4.087747
70 2025-11-01  4.545253
71 2025-12-01  4.988000


In [31]:
model_2026 = ExponentialSmoothing(
    train_2026["price"],
    trend="add",
    seasonal="add",
    seasonal_periods=12
)

fit_2026 = model_2026.fit()

In [32]:
forecast_2026 = fit_2026.forecast(12)

forecast_months_2026 = pd.date_range(
    start="2026-01-01",
    periods=12,
    freq="MS"
)

strawberry_forecast = pd.DataFrame({
    "month": forecast_months_2026,
    "price": forecast_2026.values
})

strawberry_forecast

,month,price
0,2026-01-01,4.787594
1,2026-02-01,4.390094
2,2026-03-01,3.916928
3,2026-04-01,3.982502
4,2026-05-01,3.732261
5,2026-06-01,3.483759
6,2026-07-01,3.452423
7,2026-08-01,3.728588
8,2026-09-01,3.891588
9,2026-10-01,4.190713


In [33]:
strawberry_forecast.to_csv(
    "strawberry-forecast.tsv",
    sep="\t",
    index=False
)

print("Saved strawberry-forecast.tsv")

Saved strawberry-forecast.tsv


In [34]:
check_forecast = pd.read_csv(
    "strawberry-forecast.tsv",
    sep="\t"
)

print(check_forecast)
print("\nShape:", check_forecast.shape)
print("Columns:", check_forecast.columns.tolist())

         month     price
0   2026-01-01  4.787594
1   2026-02-01  4.390094
2   2026-03-01  3.916928
3   2026-04-01  3.982502
4   2026-05-01  3.732261
5   2026-06-01  3.483759
6   2026-07-01  3.452423
7   2026-08-01  3.728588
8   2026-09-01  3.891588
9   2026-10-01  4.190713
10  2026-11-01  4.698632
11  2026-12-01  5.105960

Shape: (12, 2)
Columns: ['month', 'price']


Submit "strawberry-forecast.tsv" in Gradescope.

## Part 4: Buy Low, Freeze and Sell High

Using your 2026 forecast, analyze the profit picking different pairs of months to buy and sell strawberries.
Maximize your profit assuming that it costs &dollar;0.20 per pint to freeze the strawberries, &dollar;0.10 per pint per month to store the frozen strawberries and there is a 10% price discount from selling previously frozen strawberries.
So, if you buy a pint of strawberies for &dollar;1, freeze them, and sell them for &dollar;2 three months after buying them, then the profit is &dollar;2 * 0.9 - &dollar;1 - &dollar;0.20 - &dollar;0.10 * 3 = &dollar;0.30 per pint.
To evaluate a given pair of months, assume that you can invest &dollar;1,000,000 to cover all costs, and that you buy as many pints of strawberries as possible.

Write the results of your analysis to a file "timings.tsv" with columns for the buy_month, sell_month, pints_purchased, and expected_profit.

In [35]:
# YOUR CHANGES HERE

results = []

budget = 1_000_000
freeze_cost = 0.20
storage_cost_per_month = 0.10
frozen_discount = 0.90

for buy_idx in range(len(strawberry_forecast)):
    for sell_idx in range(buy_idx + 1, len(strawberry_forecast)):

        buy_month = strawberry_forecast.iloc[buy_idx]["month"]
        sell_month = strawberry_forecast.iloc[sell_idx]["month"]

        buy_price = strawberry_forecast.iloc[buy_idx]["price"]
        sell_price = strawberry_forecast.iloc[sell_idx]["price"]

        # Number of months strawberries are stored
        months_stored = sell_idx - buy_idx

        # Total cost per pint
        total_cost_per_pint = (
            buy_price
            + freeze_cost
            + storage_cost_per_month * months_stored
        )

        # Buy as many whole pints as the $1M budget allows
        pints_purchased = int(budget // total_cost_per_pint)

        # Revenue after 10% frozen-strawberry discount
        revenue = (
            pints_purchased
            * sell_price
            * frozen_discount
        )

        # Actual total cost
        total_cost = (
            pints_purchased
            * total_cost_per_pint
        )

        expected_profit = revenue - total_cost

        results.append({
            "buy_month": buy_month,
            "sell_month": sell_month,
            "pints_purchased": pints_purchased,
            "expected_profit": expected_profit
        })

In [36]:
timings = pd.DataFrame(results)

print(timings.shape)
timings.head()

(66, 4)


,buy_month,sell_month,pints_purchased,expected_profit
0,2026-01-01,2026-02-01,196556,-223387.644918
1,2026-01-01,2026-03-01,192767,-320447.869717
2,2026-01-01,2026-04-01,189121,-322137.816761
3,2026-01-01,2026-05-01,185611,-376522.855321
4,2026-01-01,2026-06-01,182229,-428641.077772


In [37]:
timings_sorted = timings.sort_values(
    "expected_profit",
    ascending=False
)

timings_sorted.head(10)

,buy_month,sell_month,pints_purchased,expected_profit
55,2026-07-01,2026-12-01,240823,106670.240954
50,2026-06-01,2026-12-01,233439,72740.837813
59,2026-08-01,2026-12-01,231022,61631.149748
62,2026-09-01,2026-12-01,227708,46401.432075
54,2026-07-01,2026-11-01,246765,43515.894253
49,2026-06-01,2026-11-01,239019,10758.341254
64,2026-10-01,2026-12-01,217831,1013.123175
58,2026-08-01,2026-11-01,236485,42.844851
44,2026-05-01,2026-12-01,215877,-7965.226839
61,2026-09-01,2026-11-01,233013,-14637.621250


In [38]:
best = timings.loc[timings["expected_profit"].idxmax()]

print("Best strategy:")
print("Buy month:", best["buy_month"])
print("Sell month:", best["sell_month"])
print("Pints purchased:", best["pints_purchased"])
print("Expected profit:", best["expected_profit"])

Best strategy:
Buy month: 2026-07-01 00:00:00
Sell month: 2026-12-01 00:00:00
Pints purchased: 240823
Expected profit: 106670.24095394323


In [39]:
timings.to_csv(
    "timings.tsv",
    sep="\t",
    index=False
)

print("Saved timings.tsv")

Saved timings.tsv


In [40]:
check_timings = pd.read_csv(
    "timings.tsv",
    sep="\t"
)

print(check_timings.head())
print("\nShape:", check_timings.shape)
print("Columns:", check_timings.columns.tolist())

    buy_month  sell_month  pints_purchased  expected_profit
0  2026-01-01  2026-02-01           196556   -223387.644918
1  2026-01-01  2026-03-01           192767   -320447.869717
2  2026-01-01  2026-04-01           189121   -322137.816761
3  2026-01-01  2026-05-01           185611   -376522.855321
4  2026-01-01  2026-06-01           182229   -428641.077772

Shape: (66, 4)
Columns: ['buy_month', 'sell_month', 'pints_purchased', 'expected_profit']


Submit "timings.tsv" in Gradescope.

## Part 5: Strategy Check

What is the best profit scenario according to your previous timing analysis?
How much does that profit change if the sell price is off by one standard deviation from your backtest analysis?
(Variation in the sell price is more dangerous because you can see the buy price before fully committing.)

Write the results to a file "check.tsv" with columns `best_profit` and `one_std_profit`.
To be clear, `one_std_profit` should be the number of pints bought in your best profit scenario times your backtested standard deviation of the residual.
This represents the standard deviation in revenue when selling if you explicitly assume that you buy according to the best profit scenario and your backtest standard deviation is representative of the future prices.

In [41]:
# YOUR CHANGES HERE
best_scenario = timings.loc[
    timings["expected_profit"].idxmax()
]

print(best_scenario)

buy_month          2026-07-01 00:00:00
sell_month         2026-12-01 00:00:00
pints_purchased                 240823
expected_profit          106670.240954
Name: 55, dtype: object


In [42]:
best_profit = best_scenario["expected_profit"]
pints_best = best_scenario["pints_purchased"]

print("Best profit:", best_profit)
print("Pints purchased:", pints_best)

Best profit: 106670.24095394323
Pints purchased: 240823


In [43]:
print("Backtest residual std:", residual_std)

Backtest residual std: 0.14698638142689274


In [44]:
backtest_accuracy = pd.read_csv(
    "backtest-accuracy.tsv",
    sep="\t"
)

residual_std = backtest_accuracy.loc[0, "std"]

In [45]:
one_std_profit = pints_best * residual_std

print("Best profit:", best_profit)
print("One standard deviation profit change:", one_std_profit)

Best profit: 106670.24095394323
One standard deviation profit change: 35397.701334368576


In [46]:
check = pd.DataFrame({
    "best_profit": [best_profit],
    "one_std_profit": [one_std_profit]
})

check.to_csv(
    "check.tsv",
    sep="\t",
    index=False
)

check

,best_profit,one_std_profit
0,106670.240954,35397.701334


In [47]:
pd.read_csv("check.tsv", sep="\t")

,best_profit,one_std_profit
0,106670.240954,35397.701334


Submit "check.tsv" in Gradescope.

## Part 6: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgments are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Submit "project.ipynb" in Gradescope.

Submitting via Github Repo